In [0]:
%sql
SELECT COUNT(*) AS unmatched_companies
FROM ask_sarthak.gold.fact_gl_transactions f
LEFT ANTI JOIN ask_sarthak.gold.dim_company c
    ON f.company_code = c.company_code;


GL Account

In [0]:
%sql
SELECT COUNT(*) AS unmatched_accounts
FROM ask_sarthak.gold.fact_gl_transactions f
LEFT ANTI JOIN ask_sarthak.gold.dim_gl_account a
    ON f.gl_account = a.gl_account;


Vendor

In [0]:
%sql
SELECT COUNT(*) AS unmatched_vendors
FROM ask_sarthak.gold.fact_gl_transactions f
LEFT ANTI JOIN ask_sarthak.gold.dim_vendor v
    ON f.vendor_id = v.vendor_id
WHERE f.vendor_id IS NOT NULL;


Customer

In [0]:
%sql
SELECT COUNT(*) AS unmatched_customers
FROM ask_sarthak.gold.fact_gl_transactions f
LEFT ANTI JOIN ask_sarthak.gold.dim_customer c
    ON f.customer_id = c.customer_id
WHERE f.customer_id IS NOT NULL;

In [0]:
%sql
DESCRIBE TABLE ask_sarthak.gold.fact_gl_transactions;

In [0]:
%sql
DESCRIBE TABLE ask_sarthak.gold.dim_gl_account;


What these metrics mean: Metric	Definition,
Total Debit : Sum of debit amounts,
Total Credit : Sum of credit amounts,
Net Debit : Debit minus credit,
Transaction Line Count : Number of journal lines,
Journal Entry Count : Distinct document numbers within each company and fiscal year

In [0]:
%sql
CREATE OR REPLACE VIEW ask_sarthak.analytics.vw_finance_kpis AS

SELECT
    company_code,
    fiscal_year,
    currency,

    SUM(debit_amount) AS total_debit,

    SUM(credit_amount) AS total_credit,

    SUM(debit_amount) - SUM(credit_amount) AS net_debit,

    COUNT(*) AS transaction_line_count,

    COUNT(DISTINCT document_number) AS journal_entry_count

FROM ask_sarthak.gold.fact_gl_transactions

GROUP BY
    company_code,
    fiscal_year,
    currency;


This tells us the actual classification values in your dimension. We shouldn't assume the values are ASSET, LIABILITY, REVENUE, or EXPENSE without checking.

In [0]:
%sql
SELECT
    account_type,
    COUNT(*) AS account_count
FROM ask_sarthak.gold.dim_gl_account
GROUP BY account_type
ORDER BY account_type;


Create the extended finance KPI view

Why this calculation?
- Revenue normally increases with a credit, so we calculate credit - debit.
- Expenses normally increase with a debit, so we calculate debit - credit.
These formulas calculate net balances by account type, not gross posting volume.

In [0]:
%sql
CREATE OR REPLACE VIEW ask_sarthak.analytics.vw_finance_kpis AS

SELECT
    f.company_code,
    f.fiscal_year,
    f.currency,

    SUM(f.debit_amount) AS total_debit,
    SUM(f.credit_amount) AS total_credit,

    SUM(f.debit_amount) - SUM(f.credit_amount) AS net_debit,

    COUNT(*) AS transaction_line_count,

    COUNT(DISTINCT f.document_number) AS journal_entry_count,

    SUM(
        CASE
            WHEN a.account_type = 'REVENUE'
            THEN f.credit_amount - f.debit_amount
            ELSE 0
        END
    ) AS total_revenue,

    SUM(
        CASE
            WHEN a.account_type = 'EXPENSE'
            THEN f.debit_amount - f.credit_amount
            ELSE 0
        END
    ) AS total_expenses

FROM ask_sarthak.gold.fact_gl_transactions f

LEFT JOIN ask_sarthak.gold.dim_gl_account a
    ON f.gl_account = a.gl_account

GROUP BY
    f.company_code,
    f.fiscal_year,
    f.currency;


Step 4 — Validate the KPI view

In [0]:
%sql
SELECT *
FROM ask_sarthak.analytics.vw_finance_kpis
ORDER BY company_code, fiscal_year;

Step 4 - Create the monthly finance analytics view

In [0]:
%sql
CREATE OR REPLACE VIEW ask_sarthak.analytics.vw_monthly_finance AS

SELECT
    f.company_code,
    f.fiscal_year,
    DATE_TRUNC('MONTH', f.posting_date) AS posting_month,
    f.currency,

    SUM(f.debit_amount) AS total_debit,
    SUM(f.credit_amount) AS total_credit,

    SUM(f.debit_amount) - SUM(f.credit_amount) AS net_debit,

    SUM(
        CASE
            WHEN a.account_type = 'REVENUE'
            THEN f.credit_amount - f.debit_amount
            ELSE 0
        END
    ) AS total_revenue,

    SUM(
        CASE
            WHEN a.account_type = 'EXPENSE'
            THEN f.debit_amount - f.credit_amount
            ELSE 0
        END
    ) AS total_expenses,

    COUNT(*) AS transaction_line_count,

    COUNT(DISTINCT f.document_number) AS journal_entry_count

FROM ask_sarthak.gold.fact_gl_transactions f

LEFT JOIN ask_sarthak.gold.dim_gl_account a
    ON f.gl_account = a.gl_account

GROUP BY
    f.company_code,
    f.fiscal_year,
    DATE_TRUNC('MONTH', f.posting_date),
    f.currency;


Step 5 — Validate monthly results

In [0]:
%sql
SELECT *
FROM ask_sarthak.analytics.vw_monthly_finance
WHERE company_code = '1000'
ORDER BY posting_month;


Step 6 — Test a business question

In [0]:
%sql
SELECT
    company_code,
    posting_month,
    total_revenue,
    total_expenses,
    total_revenue - total_expenses AS profit_before_other_items
FROM ask_sarthak.analytics.vw_monthly_finance
WHERE company_code = '1000'
ORDER BY posting_month;


Create your first Metric View

In [0]:
%sql
CREATE OR REPLACE VIEW ask_sarthak.analytics.mv_finance_monthly
WITH METRICS
LANGUAGE YAML
AS
$$
version: 1.1

comment: "Governed monthly financial metrics for Ask Sarthak"

source: ask_sarthak.analytics.vw_monthly_finance

dimensions:
  - name: Company
    expr: company_code
    comment: "Company code for financial reporting"
    synonyms: ["company", "entity"]

  - name: Fiscal Year
    expr: fiscal_year
    comment: "Fiscal year of the financial records"
    synonyms: ["year", "financial year"]

  - name: Posting Month
    expr: posting_month
    comment: "Month in which the transaction was posted"
    synonyms: ["month", "posting period"]

  - name: Currency
    expr: currency
    comment: "Transaction currency"

measures:
  - name: Total Debit
    expr: SUM(total_debit)
    comment: "Total debit amount"
    synonyms: ["debit", "debit amount"]

  - name: Total Credit
    expr: SUM(total_credit)
    comment: "Total credit amount"
    synonyms: ["credit", "credit amount"]

  - name: Net Debit
    expr: SUM(net_debit)
    comment: "Total debit minus total credit"

  - name: Total Revenue
    expr: SUM(total_revenue)
    comment: "Net revenue based on revenue account balances"
    synonyms: ["revenue", "sales"]

  - name: Total Expenses
    expr: SUM(total_expenses)
    comment: "Net expenses based on expense account balances"
    synonyms: ["expenses", "expenditure", "costs"]

  - name: Transaction Lines
    expr: SUM(transaction_line_count)
    comment: "Number of journal line items"

  - name: Journal Entries
    expr: SUM(journal_entry_count)
    comment: "Number of journal entries"
    synonyms: ["document count", "journal count"]
$$;


Why do we need GROUP BY?
Think of it this way:
- Dimensions determine how you want to see the data: company, year, month, currency.
- Measures determine what you want to calculate: debit, credit, revenue, expenses.

In [0]:
%sql
SELECT
    Company,
    `Fiscal Year`,
    `Posting Month`,
    Currency,

    MEASURE(`Total Debit`) AS total_debit,
    MEASURE(`Total Credit`) AS total_credit,
    MEASURE(`Net Debit`) AS net_debit,
    MEASURE(`Total Revenue`) AS total_revenue,
    MEASURE(`Total Expenses`) AS total_expenses,
    MEASURE(`Transaction Lines`) AS transaction_lines,
    MEASURE(`Journal Entries`) AS journal_entries

FROM ask_sarthak.analytics.mv_finance_monthly

GROUP BY
    Company,
    `Fiscal Year`,
    `Posting Month`,
    Currency

ORDER BY
    Company,
    `Posting Month`;


Step 7 — Test a business-level question

In [0]:
%sql
SELECT
    Company,
    MEASURE(`Total Revenue`) AS revenue,
    MEASURE(`Total Expenses`) AS expenses
FROM ask_sarthak.analytics.mv_finance_monthly
GROUP BY Company
ORDER BY Company;